# 📤 Notebook 2: Transactional Outbox + Polling Publisher

Store outbound events in an `outbox` table within the same transaction as the data change. A small **publisher** loop polls the table, ships unpublished rows to the bus, and marks them as sent. Crash-safe by construction.


## 🛠️ Setup

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```
Select the `.venv` kernel in VS Code.


## 🟩 Schema + transactional write

In [ ]:
import psycopg, json, time
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS orders')
    conn.execute('DROP TABLE IF EXISTS outbox')
    conn.execute('CREATE TABLE orders (id SERIAL PRIMARY KEY, item TEXT, total INTEGER)')
    conn.execute('''
        CREATE TABLE outbox (
            id SERIAL PRIMARY KEY,
            topic TEXT NOT NULL,
            payload JSONB NOT NULL,
            created_at TIMESTAMPTZ DEFAULT now(),
            published_at TIMESTAMPTZ
        )''')

def place_order(item, total):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            cur = conn.execute('INSERT INTO orders(item,total) VALUES (%s,%s) RETURNING id',
                               (item, total))
            order_id = cur.fetchone()[0]
            conn.execute(
                'INSERT INTO outbox(topic, payload) VALUES (%s, %s::jsonb)',
                ('order.placed', json.dumps({'order_id': order_id, 'item': item, 'total': total}))
            )
        return order_id

for it in [('book',25),('pen',5),('lamp',40)]:
    place_order(*it)

with psycopg.connect(DSN) as conn:
    print('orders:', conn.execute('SELECT * FROM orders').fetchall())
    print('outbox:', conn.execute('SELECT id, topic, payload, published_at FROM outbox').fetchall())


## 📬 Polling publisher (the BETTER design)

In [ ]:
published = []  # pretend Kafka / RabbitMQ

def publish_round(batch_size=10):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            rows = conn.execute('''
                SELECT id, topic, payload FROM outbox
                WHERE published_at IS NULL
                ORDER BY id
                LIMIT %s
                FOR UPDATE SKIP LOCKED
            ''', (batch_size,)).fetchall()
            for row_id, topic, payload in rows:
                published.append({'topic': topic, 'payload': payload})
                conn.execute('UPDATE outbox SET published_at = now() WHERE id = %s', (row_id,))
        return len(rows)

n = publish_round()
print(f'published {n} events')
for e in published: print(' ', e)


`FOR UPDATE SKIP LOCKED` lets multiple publisher workers run safely in parallel without stepping on each other.

## 🧠 Why this is correct

- The outbox row and the business row commit **atomically**. Either both exist or neither does.
- The publisher is **at-least-once** — a crash mid-publish just means we'll re-send. Pair with idempotency keys downstream.
- We never publish phantom events for rolled-back transactions.